# Run OpenMM from SAMMD output files

This notebook starts after SAMMD has written files for OpenFF/OpenMM. SAMMD builds/exports files. OpenMM runs minimization, equilibration, production, trajectories, and reporters.

Use the raw OpenMM Python API here, not a SAMMD OpenMM wrapper.

## What you need before starting

From the repository root, build the OpenFF/OpenMM files in the default pixi environment. Use a CUDA-labeled environment only when you need a specific GPU OpenMM pin. For GPU work, run `nvidia-smi` first, then choose an environment whose CUDA version is not newer than the CUDA version shown there.

```bash
pixi run sammd build sammd-project/sammd.yaml --output-dir sammd-project/outputs --overwrite
```

This should write `interchange.json`, `solvated_system.cif`, `solvated_system_pymol.pdb`, and `anchor_metadata.json` in `sammd-project/outputs/`.

## Imports

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from openff.interchange import Interchange
from openmm import LangevinMiddleIntegrator, MonteCarloAnisotropicBarostat, MonteCarloBarostat, Vec3, unit
from openmm.app import DCDReporter, Simulation, StateDataReporter
from sammd.backends.interchange_plugins import register_interchange_plugin_collection

## Path checks

Change `output_dir` if your SAMMD files are in another folder.

In [ ]:
output_dir = Path("sammd-project/outputs")
interchange_path = output_dir / "interchange.json"
trajectory_path = output_dir / "trajectory.dcd"
thermo_path = output_dir / "thermodynamics.csv"

required_paths = [interchange_path]
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    missing = ", ".join(str(path) for path in missing_paths)
    raise FileNotFoundError(f"Missing SAMMD output file(s): {missing}")

print(f"Using {interchange_path}")

## Load Interchange and export OpenMM objects

The main route is to register SAMMD's Interchange plugin collection, load `interchange.json` with OpenFF Interchange, then ask Interchange for the OpenMM `System`, topology, and positions.

In [ ]:
register_interchange_plugin_collection()
interchange = Interchange.model_validate_json(interchange_path.read_text(encoding="utf-8"))
system = interchange.to_openmm()
topology = interchange.to_openmm_topology()
positions = interchange.get_positions(include_virtual_sites=True).to_openmm()

## About Interchange reload

SAMMD stores the sulfur-metal pair overrides in a plugin collection inside `interchange.json`. Register that collection before calling `Interchange.model_validate_json` so `interchange.to_openmm()` can apply the OpenMM exceptions from the reloaded artifact.

## NVT settings and helper functions

OpenMM needs integer step counts and integer reporter intervals. Start from friendly numbers, then let helper functions do the unit math. Here we use `production_time_ns = 10.0`, `desired_trajectory_frames = 300`, and `desired_thermo_points = 1000`.

In [ ]:
temperature = 300.0 * unit.kelvin
friction = 1.0 / unit.picosecond
timestep = 2.0 * unit.femtosecond
equilibration_time_ps = 100.0
production_time_ns = 0.5
desired_trajectory_frames = 180
desired_thermo_points = 100

def steps_from_time(time, time_unit, timestep):
    """Convert a desired time to an integer number of OpenMM steps."""
    return int(round((time * time_unit / timestep)))

def interval_from_count(total_steps, desired_count):
    """Convert a desired number of saved points to an integer report interval."""
    if desired_count < 1:
        raise ValueError("desired_count must be at least 1")
    return max(1, total_steps // desired_count)

equilibration_steps = steps_from_time(equilibration_time_ps, unit.picosecond, timestep)
production_steps = steps_from_time(production_time_ns, unit.nanosecond, timestep)
trajectory_interval = interval_from_count(production_steps, desired_trajectory_frames)
thermo_interval = interval_from_count(production_steps, desired_thermo_points)

print(f"Equilibration steps: {equilibration_steps}")
print(f"Production steps: {production_steps}")
print(f"Trajectory interval: every {trajectory_interval} steps")
print(f"Thermo interval: every {thermo_interval} steps")

## Create the Simulation

NVT is the default here. In NVT, the number of particles and the volume stay fixed, and the thermostat targets the chosen temperature. The instantaneous temperature will still fluctuate.

In [ ]:
integrator = LangevinMiddleIntegrator(temperature, friction, timestep)
simulation = Simulation(topology, system, integrator)
simulation.context.setPositions(positions)

## Check initial energy

The first potential energy is often a large positive number. That is common for a starting structure and often motivates minimization. The real first check is that the energy is finite, not `nan` or `inf`.

In [ ]:
initial_state = simulation.context.getState(getEnergy=True)
initial_energy = initial_state.getPotentialEnergy()
initial_energy_value = initial_energy.value_in_unit(unit.kilojoule_per_mole)
if not math.isfinite(initial_energy_value):
    raise ValueError(f"Initial potential energy is not finite: {initial_energy}")
print(f"Initial potential energy: {initial_energy}")

## Minimize energy

Use the simple `simulation.minimizeEnergy()` call for this beginner workflow.

In [ ]:
simulation.minimizeEnergy()
minimized_state = simulation.context.getState(getEnergy=True)
print(f"Minimized potential energy: {minimized_state.getPotentialEnergy()}")

## NVT equilibration

In [ ]:
simulation.context.setVelocitiesToTemperature(temperature)
simulation.step(equilibration_steps)

## Add reporters and run NVT production

`DCDReporter` writes the trajectory. `StateDataReporter` writes a CSV table with energies, temperature, and speed.

In [ ]:
simulation.reporters.append(DCDReporter(str(trajectory_path), trajectory_interval))
simulation.reporters.append(
    StateDataReporter(
        str(thermo_path),
        thermo_interval,
        step=True,
        time=True,
        potentialEnergy=True,
        kineticEnergy=True,
        totalEnergy=True,
        temperature=True,
        speed=True,
        separator=",",
    )
)
simulation.step(production_steps)

## Load and plot the thermo CSV

In [ ]:
thermo = pd.read_csv(thermo_path)
thermo.head()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(thermo["Time (ps)"], thermo["Potential Energy (kJ/mole)"])
plt.xlabel("Time (ps)")
plt.ylabel("Potential energy (kJ/mol)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(thermo["Time (ps)"], thermo["Temperature (K)"])
plt.xlabel("Time (ps)")
plt.ylabel("Temperature (K)")
plt.tight_layout()
plt.show()

## Optional NPT note

Use NPT only when pressure control makes sense. Choose one of these examples, not both. For bulk systems, add `MonteCarloBarostat` before creating `Simulation`. For a slab or interface, use `MonteCarloAnisotropicBarostat` before creating `Simulation` with `x` and `y` fixed and `z` allowed. A barostat does not replace the thermostat, so keep `LangevinMiddleIntegrator`.

Bulk fluid example:

```python
pressure = 1.0 * unit.atmosphere
system.addForce(MonteCarloBarostat(pressure, temperature))
simulation = Simulation(topology, system, integrator)
```

Slab or interface example:

```python
pressure = 1.0 * unit.atmosphere

system.addForce(
    MonteCarloAnisotropicBarostat(
        Vec3(1.0, 1.0, 1.0) * unit.atmosphere,
        temperature,
        False,
        False,
        True,
    )
)
simulation = Simulation(topology, system, integrator)
```

## View the trajectory in PyMOL

Open the structure first, then load the DCD into the same object:

```pymol
load sammd-project/outputs/solvated_system_pymol.pdb, sammd_system
load_traj sammd-project/outputs/trajectory.dcd, sammd_system
```